# Creación de un modelo para detección de mensajes de spam

## Leer los datos y explorarlos

In [1]:
import pandas as pd

In [2]:
# Cargar el dataset inglés de spam

df = pd.read_csv('spam.csv', encoding='latin-1')
df = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)
df = df.rename(columns={'v1': 'label', 'v2': 'text'})

In [3]:
# Cargar el dataset español de spam

df_es = pd.read_csv('spam_es.csv', encoding='latin-1')

In [4]:
df = df_es

In [5]:
df

,text,label
0,Compra ahora y recibe un descuento especial,ham
1,Haz clic aqui para ganar un premio,spam
2,Tu ordenador tiene un virus,spam
3,Descubre como perder peso rapidamente,spam
4,Necesitas ayuda con tu tarea,ham
...,...,...
993,Sorprende a tu pareja con este perfume,spam
994,Resolveremos tu caso a la brevedad,ham
995,Invierte en bienes raÃ­ces con expertos,spam
996,Hola evaluemos tu perfil profesional,ham


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 998 entries, 0 to 997
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    998 non-null    object
 1   label   998 non-null    object
dtypes: object(2)
memory usage: 15.7+ KB


In [7]:
# Distribución de las clases
print(df['label'].value_counts())

label
spam    521
ham     477
Name: count, dtype: int64


In [8]:
df.tail(10)

,text,label
988,Gracias por pagar puntualmente,ham
989,Descubre las claves de un negocio exitoso,spam
990,Revisa y aprueba el contrato digital,ham
991,Pide tu muestra mÃ©dica gratuita,spam
992,Rastrea tu envÃ­o con este cÃ³digo,ham
993,Sorprende a tu pareja con este perfume,spam
994,Resolveremos tu caso a la brevedad,ham
995,Invierte en bienes raÃ­ces con expertos,spam
996,Hola evaluemos tu perfil profesional,ham
997,Ahorra en grande con nuestra cuenta VIP,spam


## Crear y entrenar el modelo

### TF-IDF

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [10]:
# Dividir el dataset en entrenamiento y prueba
X = df['text']
y = df['label']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [12]:
# Pipeline de procesamiento y clasificación
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [13]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer(stop_words='english')),
                ('clf', RandomForestClassifier(random_state=42))])

In [14]:
# Imprimir el classification report
print(classification_report(y_test, pipeline.predict(X_test)))

              precision    recall  f1-score   support

         ham       0.84      0.88      0.86        52
        spam       0.87      0.81      0.84        48

    accuracy                           0.85       100
   macro avg       0.85      0.85      0.85       100
weighted avg       0.85      0.85      0.85       100



In [15]:
mensaje = "Eres el ganador de un premio, haz clic aquí para reclamarlo"

In [16]:
# Predicción del mensaje de prueba:
prediccion = pipeline.predict([mensaje])[0]
print(f"El mensaje es clasificado como: {prediccion}")

El mensaje es clasificado como: spam


## Guardar el modelo

In [17]:
import pickle

# Guardar el modelo entrenado
with open('modelo_spam.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

## Probar la API

In [ ]:
import requests
mensaje = "Eres el ganador de un premio, haz clic aquí para reclamarlo"

response = requests.post('http://127.0.0.1:8000/predict', json={'text': mensaje})

print (response.json())

{'label': 'spam', 'is_spam': True, 'spam_probability': 0.65}
